# Final pipeline, end to end

**One job:** reproduce the final solution from a clean Colab T4 runtime: a YOLO26l detector (`l_ps`) proposes boxes and one box-prompted U-Net (`b5_sym_s2`) segments each filament. Both checkpoints are in `weights/`.

**Data:** put the competition download at `/content/kaggle_data`, or leave it to the next cell, which needs a Kaggle token at `~/.kaggle/access_token` from an account that joined the competition. The images are not in this repo because they cannot be redistributed.

The demo takes under a minute once the data is in place. Validation (~3 min) and test (~5 min) are behind flags that default to `False`. Training is summarized at the end.

In [ ]:
!git clone -q https://github.com/franciscomendez114/solar-filament-segmentation.git /content/solar-filament-segmentation
%cd /content/solar-filament-segmentation
%pip install -q ultralytics==8.4.191 pycocotools segmentation-models-pytorch==0.5.0 kaggle==2.2.4
!test -d /content/kaggle_data || (kaggle competitions download -c filament-segmentation-2026 -p /content && unzip -q /content/filament-segmentation-2026.zip -d /content/kaggle_data)

In [ ]:
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pycocotools import mask as mask_utils
from ultralytics import YOLO
from ultralytics.utils import LOGGER

LOGGER.setLevel("ERROR")  # two deprecation warnings per image otherwise
sys.path.insert(0, "/content/solar-filament-segmentation")
from src.config import DEFAULT_PIPELINE_SETTINGS
from src.data import load_records, verify_or_write_split
from src.inference import cache_proposals, create_submission, fuse_boxes, predict_instances
from src.metrics import aggregate_official, annotation_to_rle, official_match_counts
from src.models import RotationTTA, load_checkpoint

RUN_VALIDATION = False  # reproduces official PQ 0.4666 on the frozen 106-image split
RUN_TEST = False  # writes submission.csv for the 180 test images
DETECTOR_CONFIDENCE, MASK_THRESHOLD = 0.25, 0.45

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
detector = YOLO("weights/l_ps.pt")
unet = RotationTTA(load_checkpoint("weights/b5_sym_s2.pt", "b5", device)[0]).eval()  # 8-way rotation and flip TTA
settings = {**DEFAULT_PIPELINE_SETTINGS, "yolo_confidence": 0.01, "yolo_image_size": 1280}

train_dir = next(Path("/content/kaggle_data").rglob("train_images")).parent
test_paths = sorted(next(Path("/content/kaggle_data").rglob("test_images")).glob("*.jpeg"))
_, records = load_records(train_dir / "MAGFiLO_1.0_Annotations_kaggle2026_train.json", train_dir / "train_images")
_, _, validation_records = verify_or_write_split(records, Path("data/processed/split_manifest.json"))
print(device, len(validation_records), "validation records,", len(test_paths), "test images")


def final_boxes(image_paths):
    # Stage 1: confidence >= 0.25, after the fusion the ensemble used, which with one detector only merges near-identical boxes
    proposals = {}
    for stem, candidates in cache_proposals(detector, image_paths, device, settings).items():
        boxes, scores = fuse_boxes([candidates], [1.0], 0.40, 0.01)
        keep = scores >= DETECTOR_CONFIDENCE
        proposals[stem] = (np.asarray(boxes[keep], dtype=np.float32).reshape(-1, 4), np.asarray(scores[keep], dtype=np.float32))
    return proposals

## Demo: one validation image

Left: the detector's boxes. Right: the final non-overlapping masks, drawn dilated so thin filaments stay visible.

In [ ]:
import cv2

record = validation_records[0]
path = record["image_path"]
boxes, scores = final_boxes([path])[Path(path).stem]
masks, _ = predict_instances(path, None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS, proposals=(boxes, scores))
image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
overlay = np.zeros((*image.shape, 4))
for mask in masks:
    overlay[cv2.dilate(mask.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0] = (0.1, 0.9, 0.4, 0.9)
figure, axes = plt.subplots(1, 2, figsize=(12, 6))
for axis in axes:
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.axis("off")
for x1, y1, x2, y2 in boxes:
    axes[0].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="orange", linewidth=1))
axes[0].set_title(f"{len(boxes)} detector boxes")
axes[1].imshow(overlay)
axes[1].set_title(f"{len(masks)} final masks")
plt.show()

## Validation: official PQ

Scores each annotator record separately and pools the counts, as the organizers' code does. Expected: PQ 0.4666, TP 828, FP 360, FN 430.

In [ ]:
if RUN_VALIDATION:
    by_image = defaultdict(list)
    for record in validation_records:
        by_image[record["physical_id"]].append(record)
    proposals = final_boxes([group[0]["image_path"] for group in by_image.values()])
    rows = []
    for physical_id, group in by_image.items():
        masks, _ = predict_instances(group[0]["image_path"], None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS, proposals=proposals[physical_id])
        predicted = [mask_utils.encode(np.asfortranarray(mask)) for mask in masks]
        for record in group:
            targets = [annotation_to_rle(annotation, record["height"], record["width"]) for annotation in record["annotations"]]
            rows.append({"physical_id": physical_id, "annotator_image_id": record["annotator_image_id"], "predicted_instances": len(masks), **official_match_counts(predicted, targets)})
    print(aggregate_official(pd.DataFrame(rows)))

## Test: Kaggle submission

Writes `submission.csv` after checking for duplicate ids and overlapping masks. This pipeline scored 0.39 on the public leaderboard.

In [ ]:
if RUN_TEST:
    submission, _ = create_submission(test_paths, None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS, final_boxes(test_paths))
    submission.to_csv("submission.csv", index=False)
    print(len(submission), "masks in submission.csv")

## Training

Training ran on Colab GPUs through the scripts below, not in this notebook: the final detector took about two hours on an A100.

1. **Baseline detectors and U-Net:** notebooks `02` and `03`, then `scripts/detector_train_job.py` for the YOLO11m member.
2. **Unlabeled pool:** `scripts/gong_pool.py` picks and downloads 1,000 GONG frames (`data/processed/gong_pool.json`), and `scripts/check_gong_pool.py` proves none is a test image.
3. **Teacher labels:** `scripts/launch_pseudo.py` fuses the 24 teacher views at score >= 0.30. The labels used are in `data/processed/pseudo_boxes.json`.
4. **Final detector:** `scripts/detector_train_job.py` with `{"weights": "yolo26l.pt", "imgsz": 1280, "batch": 8, "mosaic": 1.0, "flipud": 0.5, "pseudo": true}`.
5. **Final U-Net:** `scripts/unet_train_job.py`, the `b5_sym_s2` run, selected on official PQ.